In [1]:
from __future__ import annotations

import io
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd
import requests
from Bio import Entrez, SearchIO, SeqIO
from Bio.Blast import NCBIWWW
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
BASE_DIR = Path(r"D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search")
OUTPUT_DIR = BASE_DIR / "blast_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


@dataclass
class BlastConfig:
    # Engine selection
    mode: str = "auto"  # auto|local|remote
    remote_provider: str = "ncbi_qblast"
    local_program: str = "blastp"

    # Local BLAST
    local_db_path: Optional[str] = None
    db_version_label: str = "unknown"
    num_threads: int = 4

    # Search constraints
    program: str = "blastp"
    database: str = "nr"  # remote fallback DB
    evalue: float = 1e-5
    max_hits: int = 500
    max_hsps: int = 1

    # Post-filters
    min_identity: float = 30.0
    min_query_coverage: float = 50.0
    min_subject_coverage: float = 0.0
    min_alignment_length: int = 80
    min_bit_score: float = 50.0

    # Length filters
    query_len_min: Optional[int] = None
    query_len_max: Optional[int] = None
    subject_len_min: Optional[int] = None
    subject_len_max: Optional[int] = None

    # Taxonomy / organism filters
    tax_include: List[str] = field(default_factory=list)
    tax_exclude: List[str] = field(default_factory=list)
    organism_include: List[str] = field(default_factory=list)
    organism_exclude: List[str] = field(default_factory=list)
    use_regex_for_organism: bool = False

    # Pfam filters (post-enrichment)
    enable_pfam_enrichment: bool = True
    pfam_include: List[str] = field(default_factory=list)
    pfam_exclude: List[str] = field(default_factory=list)

    # Cross-query deduplication
    deduplicate_across_queries: bool = False
    duplicate_key: str = "protein_id"  # protein_id | subject_id | clean_subject_id

    # Networking and reproducibility
    entrez_email: str = "phitro@bu.edu"
    entrez_api_key: Optional[str] = None
    ncbi_sleep_sec: float = 0.11
    uniprot_sleep_sec: float = 0.08
    request_timeout_sec: int = 30
    max_retries: int = 3
    backoff_base: float = 1.5

    # Output
    write_filter_audit_csv: bool = True
    output_prefix: str = "blast_run"

    def validate(self) -> None:
        if self.mode not in {"auto", "local", "remote"}:
            raise ValueError(f"Invalid mode={self.mode}. Expected auto|local|remote")
        if self.program not in {"blastp", "blastn"}:
            raise ValueError(f"Unsupported program={self.program}. Expected blastp|blastn")
        if self.min_identity < 0 or self.min_identity > 100:
            raise ValueError("min_identity must be between 0 and 100")
        if self.min_query_coverage < 0 or self.min_query_coverage > 100:
            raise ValueError("min_query_coverage must be between 0 and 100")
        if self.max_hits <= 0:
            raise ValueError("max_hits must be > 0")
        if self.mode == "local" and not self.local_db_path:
            raise ValueError("local_db_path is required in local mode")


@dataclass
class QueryInput:
    query_id: str
    input_id: Optional[str] = None  # UniProt/NCBI id
    sequence: Optional[str] = None  # raw sequence if user already has it
    organism_hint: Optional[str] = None
    taxon_hint: Optional[str] = None


@dataclass
class QueryObject:
    query_id: str
    query_sequence: str
    query_length: int
    source_db: str
    source_id: Optional[str]
    organism_hint: Optional[str]
    taxon_hint: Optional[str]


@dataclass
class HitRecord:
    query_id: str
    subject_id: str
    percent_identity: float
    evalue: float
    bit_score: float
    alignment_length: int
    query_coverage: float
    subject_coverage: float
    taxid: Optional[str]
    organism_name: Optional[str]
    subject_title: Optional[str]
    query_length: Optional[int]
    subject_length: Optional[int]
    engine: str
    db_label: str
    passed_filters: bool = False
    filter_reason: str = ""
    pfam_ids: List[str] = field(default_factory=list)
    pfam_names: List[str] = field(default_factory=list)
    source_query_id: Optional[str] = None  # which query this hit comes from
    cross_query_duplicate_ids: List[str] = field(default_factory=list)  # other query IDs with same protein


def now_str() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def clean_sequence(seq: str) -> str:
    return re.sub(r"\s+", "", seq or "").strip().upper()


def as_hit_dict(hit: HitRecord) -> Dict[str, Any]:
    d = asdict(hit)
    d["pfam_ids"] = ";".join(hit.pfam_ids)
    d["pfam_names"] = ";".join(hit.pfam_names)
    d["cross_query_duplicate_ids"] = ";".join(hit.cross_query_duplicate_ids)
    return d


def _request_with_retry(
    method: str,
    url: str,
    config: BlastConfig,
    *,
    params: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
) -> requests.Response:
    last_exc: Optional[Exception] = None
    for attempt in range(1, config.max_retries + 1):
        try:
            resp = requests.request(
                method,
                url,
                params=params,
                headers=headers,
                timeout=config.request_timeout_sec,
            )
            resp.raise_for_status()
            return resp
        except Exception as exc:
            last_exc = exc
            if attempt == config.max_retries:
                break
            sleep_time = config.backoff_base ** (attempt - 1)
            time.sleep(sleep_time)
    raise RuntimeError(f"HTTP request failed for {url}: {last_exc}")


print("Scaffold ready: dataclasses, config, shared helpers.")

Scaffold ready: dataclasses, config, shared helpers.


In [2]:
# -----------------------------------------------------------------------------
# ID and sequence resolution + identifier enrichment helpers
# -----------------------------------------------------------------------------
UNIPROT_ACCESSION_RE = re.compile(r"^[OPQ][0-9][A-Z0-9]{3}[0-9]$|^[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9]$")
WP_RE = re.compile(r"^WP_\d+(?:\.\d+)?$")
NCBI_PROT_RE = re.compile(r"^(?:NP|XP|YP|WP|AP)_\d+(?:\.\d+)?$")
NUCCORE_LIKE_RE = re.compile(r"^(?:NC|NZ|CM|CP|NG|XM|XR)_\d+(?:\.\d+)?$")
ASSEMBLY_RE = re.compile(r"^GC[AF]_\d+(?:\.\d+)?$")

_uniprot_cache: Dict[str, Dict[str, Any]] = {}
_ncbi_protein_cache: Dict[str, str] = {}
_pfam_cache: Dict[str, Dict[str, Any]] = {}
_protein_to_nuccore_cache: Dict[str, Optional[str]] = {}
_assembly_to_nuccore_cache: Dict[str, Optional[str]] = {}
_id_enrichment_cache: Dict[str, Dict[str, Any]] = {}
_ncbi_protein_meta_cache: Dict[str, Dict[str, Any]] = {}
_nuccore_record_cache: Dict[str, Optional[Any]] = {}


def _is_uniprot_accession(value: str) -> bool:
    return bool(UNIPROT_ACCESSION_RE.match((value or "").strip().upper()))


def _is_nuccore_like(value: Optional[str]) -> bool:
    if not value:
        return False
    return bool(NUCCORE_LIKE_RE.match(str(value).strip().upper()))


def _is_assembly_accession(value: Optional[str]) -> bool:
    if not value:
        return False
    return bool(ASSEMBLY_RE.match(str(value).strip().upper()))


def clean_hit_subject_id(raw_subject_id: str) -> str:
    s = (raw_subject_id or "").strip()
    if not s:
        return s

    # Typical formats:
    # ref|WP_214186582.1| -> WP_214186582.1
    # sp|Q9ABC1|NAME -> Q9ABC1
    # tr|A0A...|NAME -> A0A...
    if "|" in s:
        parts = [p.strip() for p in s.split("|")]
        if len(parts) >= 2 and parts[1]:
            return parts[1]

    return s


def _extract_accession_from_coded_by(coded_by: str) -> Optional[str]:
    text = str(coded_by or "")
    if not text:
        return None

    # Handles examples like:
    # NZ_CP012345.1:123..456
    # complement(NZ_CP012345.1:123..456)
    # join(NZ_ABCD01000001.1:1..200, ...)
    m = re.search(r"([A-Z]{1,4}_[A-Z0-9]+\.\d+)", text)
    if m:
        return m.group(1)
    return None


def _fetch_uniprot_json(accession_or_token: str, config: BlastConfig) -> Optional[Dict[str, Any]]:
    token = (accession_or_token or "").strip()
    if not token:
        return None

    cache_key = token.upper()
    if cache_key in _uniprot_cache:
        return _uniprot_cache[cache_key]

    # 1) direct accession endpoint
    direct_url = f"https://rest.uniprot.org/uniprotkb/{token}.json"
    try:
        resp = _request_with_retry("GET", direct_url, config)
        data = resp.json()
        _uniprot_cache[cache_key] = data
        time.sleep(config.uniprot_sleep_sec)
        return data
    except Exception:
        pass

    # 2) fallback search endpoint (important for WP_ cross refs)
    search_url = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": f"xref:{token}", "format": "json", "size": 1}
    try:
        resp = _request_with_retry("GET", search_url, config, params=params)
        payload = resp.json() or {}
        results = payload.get("results") or []
        data = results[0] if results else None
        if data:
            _uniprot_cache[cache_key] = data
        time.sleep(config.uniprot_sleep_sec)
        return data
    except Exception:
        return None


def _extract_uniprot_sequence(uniprot_json: Dict[str, Any]) -> Optional[str]:
    if not uniprot_json:
        return None
    seq_obj = uniprot_json.get("sequence") or {}
    value = seq_obj.get("value")
    return clean_sequence(value) if value else None


def _extract_uniprot_bundle(uniprot_json: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not uniprot_json:
        return {
            "uniprot_accession": None,
            "gene_names": [],
            "orf_names": [],
            "ordered_locus_names": [],
            "refseq_ids": [],
            "embl_ids": [],
        }

    genes: List[str] = []
    orf_names: List[str] = []
    ordered_locus_names: List[str] = []

    for g in uniprot_json.get("genes", []) or []:
        gname = ((g.get("geneName") or {}).get("value") or "").strip()
        if gname:
            genes.append(gname)

        for item in g.get("synonyms") or []:
            val = str(item.get("value", "")).strip()
            if val:
                genes.append(val)

        for item in g.get("orfNames") or []:
            val = str(item.get("value", "")).strip()
            if val:
                orf_names.append(val)

        for item in g.get("orderedLocusNames") or []:
            val = str(item.get("value", "")).strip()
            if val:
                ordered_locus_names.append(val)

    refseq_ids: List[str] = []
    embl_ids: List[str] = []

    for xref in uniprot_json.get("uniProtKBCrossReferences", []) or []:
        db = str(xref.get("database", "")).strip().lower()
        xid = str(xref.get("id", "")).strip()
        if not xid:
            continue
        if db == "refseq":
            refseq_ids.append(xid)
        elif db in {"embl", "ena"}:
            embl_ids.append(xid)

    return {
        "uniprot_accession": uniprot_json.get("primaryAccession"),
        "gene_names": sorted(set(genes)),
        "orf_names": sorted(set(orf_names)),
        "ordered_locus_names": sorted(set(ordered_locus_names)),
        "refseq_ids": sorted(set(refseq_ids)),
        "embl_ids": sorted(set(embl_ids)),
    }


def _fetch_ncbi_protein_fasta(protein_id: str, config: BlastConfig) -> Optional[str]:
    pid = (protein_id or "").strip()
    if not pid:
        return None
    if pid in _ncbi_protein_cache:
        return _ncbi_protein_cache[pid]

    Entrez.email = config.entrez_email
    if config.entrez_api_key:
        Entrez.api_key = config.entrez_api_key

    last_exc: Optional[Exception] = None
    for attempt in range(1, config.max_retries + 1):
        try:
            with Entrez.efetch(db="protein", id=pid, rettype="fasta", retmode="text") as handle:
                text = handle.read()
            lines = [ln.strip() for ln in text.splitlines() if ln.strip() and not ln.startswith(">")]
            seq = clean_sequence("".join(lines))
            if seq:
                _ncbi_protein_cache[pid] = seq
                time.sleep(config.ncbi_sleep_sec)
                return seq
            return None
        except Exception as exc:
            last_exc = exc
            if attempt < config.max_retries:
                time.sleep(config.backoff_base ** (attempt - 1))
    print(f"[WARN] NCBI protein fetch failed for {pid}: {last_exc}")
    return None


def _resolve_nuccore_from_assembly(assembly_accession: str, config: BlastConfig) -> Optional[str]:
    asm = (assembly_accession or "").strip()
    if not asm:
        return None
    if asm in _assembly_to_nuccore_cache:
        return _assembly_to_nuccore_cache[asm]

    Entrez.email = config.entrez_email
    if config.entrez_api_key:
        Entrez.api_key = config.entrez_api_key

    try:
        with Entrez.esearch(db="assembly", term=f"{asm}[Assembly Accession]", retmax=1) as handle:
            search = Entrez.read(handle)
        ids = search.get("IdList", [])
        if not ids:
            _assembly_to_nuccore_cache[asm] = None
            return None

        with Entrez.elink(dbfrom="assembly", db="nuccore", id=ids[0]) as handle:
            link_data = Entrez.read(handle)

        nuccore_ids: List[str] = []
        for entry in link_data:
            for linkset in entry.get("LinkSetDb", []):
                for link in linkset.get("Link", []):
                    lid = str(link.get("Id", "")).strip()
                    if lid:
                        nuccore_ids.append(lid)

        if not nuccore_ids:
            _assembly_to_nuccore_cache[asm] = None
            return None

        with Entrez.esummary(db="nuccore", id=nuccore_ids[0], retmode="xml") as handle:
            summary = Entrez.read(handle)
        doc = (summary or [None])[0] or {}
        accession = str(doc.get("Caption", "")).strip() or None
        _assembly_to_nuccore_cache[asm] = accession
        time.sleep(config.ncbi_sleep_sec)
        return accession
    except Exception:
        _assembly_to_nuccore_cache[asm] = None
        return None


def _resolve_nuccore_from_protein(protein_id: str, config: BlastConfig) -> Optional[str]:
    pid = (protein_id or "").strip()
    if not pid:
        return None
    if pid in _protein_to_nuccore_cache:
        return _protein_to_nuccore_cache[pid]

    Entrez.email = config.entrez_email
    if config.entrez_api_key:
        Entrez.api_key = config.entrez_api_key

    try:
        with Entrez.elink(dbfrom="protein", db="nuccore", id=pid) as handle:
            link_data = Entrez.read(handle)

        nuccore_ids: List[str] = []
        for entry in link_data:
            for linkset in entry.get("LinkSetDb", []):
                for link in linkset.get("Link", []):
                    lid = str(link.get("Id", "")).strip()
                    if lid:
                        nuccore_ids.append(lid)

        nuccore_ids = list(dict.fromkeys(nuccore_ids))
        if not nuccore_ids:
            _protein_to_nuccore_cache[pid] = None
            return None

        with Entrez.esummary(db="nuccore", id=nuccore_ids[0], retmode="xml") as handle:
            summary = Entrez.read(handle)
        doc = (summary or [None])[0] or {}
        accession = str(doc.get("Caption", "")).strip() or None
        _protein_to_nuccore_cache[pid] = accession
        time.sleep(config.ncbi_sleep_sec)
        return accession
    except Exception:
        _protein_to_nuccore_cache[pid] = None
        return None


def _fetch_ncbi_protein_metadata(protein_id: str, config: BlastConfig) -> Dict[str, Any]:
    pid = (protein_id or "").strip()
    if not pid:
        return {"locus_tag": None, "gene": None, "coded_by_nuccore": None, "assembly": None}
    if pid in _ncbi_protein_meta_cache:
        return _ncbi_protein_meta_cache[pid]

    meta = {"locus_tag": None, "gene": None, "coded_by_nuccore": None, "assembly": None}

    Entrez.email = config.entrez_email
    if config.entrez_api_key:
        Entrez.api_key = config.entrez_api_key

    try:
        with Entrez.efetch(db="protein", id=pid, rettype="gb", retmode="text") as handle:
            rec = SeqIO.read(handle, "genbank")

        for feat in rec.features:
            if feat.type == "CDS":
                qualifiers = feat.qualifiers or {}
                locus_vals = qualifiers.get("locus_tag", [])
                gene_vals = qualifiers.get("gene", [])
                coded_by_vals = qualifiers.get("coded_by", [])

                if locus_vals and not meta["locus_tag"]:
                    meta["locus_tag"] = str(locus_vals[0]).strip()
                if gene_vals and not meta["gene"]:
                    meta["gene"] = str(gene_vals[0]).strip()
                if coded_by_vals and not meta["coded_by_nuccore"]:
                    acc = _extract_accession_from_coded_by(str(coded_by_vals[0]))
                    if acc:
                        meta["coded_by_nuccore"] = acc

        for dbx in rec.dbxrefs or []:
            if dbx.upper().startswith("ASSEMBLY:"):
                asm = dbx.split(":", 1)[1].strip()
                if asm:
                    meta["assembly"] = asm
                    break

        _ncbi_protein_meta_cache[pid] = meta
        time.sleep(config.ncbi_sleep_sec)
        return meta
    except Exception:
        _ncbi_protein_meta_cache[pid] = meta
        return meta


def _fetch_nuccore_record(nuccore_accession: str, config: BlastConfig) -> Optional[Any]:
    acc = clean_hit_subject_id(nuccore_accession)
    if not acc:
        return None
    if acc in _nuccore_record_cache:
        return _nuccore_record_cache[acc]

    Entrez.email = config.entrez_email
    if config.entrez_api_key:
        Entrez.api_key = config.entrez_api_key

    try:
        with Entrez.efetch(db="nuccore", id=acc, rettype="gb", retmode="text") as handle:
            rec = SeqIO.read(handle, "genbank")
        _nuccore_record_cache[acc] = rec
        time.sleep(config.ncbi_sleep_sec)
        return rec
    except Exception:
        _nuccore_record_cache[acc] = None
        return None


def _find_locus_tag_in_nuccore(
    nuccore_accession: str,
    protein_id: str,
    candidate_names: Sequence[str],
    config: BlastConfig,
) -> Optional[str]:
    rec = _fetch_nuccore_record(nuccore_accession, config)
    if rec is None:
        return None

    pid_clean = clean_hit_subject_id(protein_id).upper()
    name_set = {str(n).strip().upper() for n in candidate_names if str(n).strip()}

    for feat in rec.features:
        if feat.type != "CDS":
            continue
        q = feat.qualifiers or {}

        # Strongest rule: direct protein_id match.
        for qpid in q.get("protein_id", []):
            if clean_hit_subject_id(str(qpid)).upper() == pid_clean and pid_clean:
                locus_vals = q.get("locus_tag", [])
                if locus_vals:
                    return str(locus_vals[0]).strip()

        # Backup rule: match gene/product/old_locus_tag with known candidates.
        if name_set:
            field_tokens: List[str] = []
            for key in ("gene", "locus_tag", "old_locus_tag", "product"):
                for v in q.get(key, []):
                    token = str(v).strip().upper()
                    if token:
                        field_tokens.append(token)
            if set(field_tokens) & name_set:
                locus_vals = q.get("locus_tag", [])
                if locus_vals:
                    return str(locus_vals[0]).strip()

    return None


def _pick_locus_tag(bundle: Dict[str, Any]) -> Optional[str]:
    for key in ("ordered_locus_names", "orf_names", "gene_names"):
        vals = bundle.get(key) or []
        for v in vals:
            token = str(v).strip()
            if token:
                return token
    return None


def _pick_genome_accession(bundle: Dict[str, Any], protein_id: str, config: BlastConfig) -> Optional[str]:
    refseq_ids = bundle.get("refseq_ids") or []
    embl_ids = bundle.get("embl_ids") or []

    for xid in refseq_ids:
        token = clean_hit_subject_id(xid)
        if _is_nuccore_like(token) or _is_assembly_accession(token):
            if _is_assembly_accession(token):
                mapped = _resolve_nuccore_from_assembly(token, config)
                return mapped or token
            return token

    for xid in embl_ids:
        token = clean_hit_subject_id(xid)
        if _is_nuccore_like(token):
            return token

    if protein_id:
        from_elink = _resolve_nuccore_from_protein(protein_id, config)
        if from_elink:
            return clean_hit_subject_id(from_elink)

    return None


def enrich_hit_identifiers(hit: HitRecord, config: BlastConfig) -> Dict[str, Any]:
    raw_subject = str(hit.subject_id or "").strip()
    clean_subject = clean_hit_subject_id(raw_subject)

    if clean_subject in _id_enrichment_cache:
        return _id_enrichment_cache[clean_subject]

    protein_id = clean_subject
    uniprot_json = _fetch_uniprot_json(clean_subject, config)
    bundle = _extract_uniprot_bundle(uniprot_json)

    uniprot_acc = bundle.get("uniprot_accession")
    locus_tag = _pick_locus_tag(bundle)
    genome_accession = _pick_genome_accession(bundle, protein_id, config)

    # Special-case resolver step 1:
    # Use NCBI protein record metadata when UniProt/crossrefs are incomplete.
    if not locus_tag or not genome_accession:
        ncbi_meta = _fetch_ncbi_protein_metadata(protein_id, config)
        locus_tag = locus_tag or ncbi_meta.get("locus_tag") or ncbi_meta.get("gene")

        coded_by_acc = ncbi_meta.get("coded_by_nuccore")
        if not genome_accession and coded_by_acc:
            genome_accession = coded_by_acc

        if not genome_accession and ncbi_meta.get("assembly"):
            asm = str(ncbi_meta.get("assembly"))
            mapped = _resolve_nuccore_from_assembly(asm, config)
            genome_accession = mapped or asm

    # Special-case resolver step 2:
    # Genome feature search in nuccore record for remaining locus-tag gaps.
    if genome_accession and not locus_tag:
        candidate_names = []
        candidate_names.extend(bundle.get("ordered_locus_names") or [])
        candidate_names.extend(bundle.get("orf_names") or [])
        candidate_names.extend(bundle.get("gene_names") or [])
        locus_from_features = _find_locus_tag_in_nuccore(
            genome_accession,
            protein_id,
            candidate_names,
            config,
        )
        locus_tag = locus_from_features or locus_tag

    enriched = {
        "clean_subject_id": clean_subject,
        "protein_id": protein_id,
        "uniprot_accession": uniprot_acc,
        "locus_tag_candidate": locus_tag,
        "genome_accession_candidate": genome_accession,
        "gene_names": ";".join(bundle.get("gene_names") or []),
        "orf_names": ";".join(bundle.get("orf_names") or []),
        "ordered_locus_names": ";".join(bundle.get("ordered_locus_names") or []),
        "refseq_ids": ";".join(bundle.get("refseq_ids") or []),
        "embl_ids": ";".join(bundle.get("embl_ids") or []),
    }
    _id_enrichment_cache[clean_subject] = enriched
    return enriched


def resolve_query_input(q: QueryInput, config: BlastConfig) -> QueryObject:
    if q.sequence:
        seq = clean_sequence(q.sequence)
        if not seq:
            raise ValueError(f"Query {q.query_id}: sequence is empty after cleaning")
        return QueryObject(
            query_id=q.query_id,
            query_sequence=seq,
            query_length=len(seq),
            source_db="raw",
            source_id=q.input_id,
            organism_hint=q.organism_hint,
            taxon_hint=q.taxon_hint,
        )

    token = (q.input_id or "").strip()
    if not token:
        raise ValueError(f"Query {q.query_id}: provide either sequence or input_id")

    uniprot_json = _fetch_uniprot_json(token, config)
    if uniprot_json:
        seq = _extract_uniprot_sequence(uniprot_json)
        if seq:
            return QueryObject(
                query_id=q.query_id,
                query_sequence=seq,
                query_length=len(seq),
                source_db="uniprot",
                source_id=token,
                organism_hint=q.organism_hint,
                taxon_hint=q.taxon_hint,
            )

    if WP_RE.match(token) or NCBI_PROT_RE.match(token):
        seq = _fetch_ncbi_protein_fasta(token, config)
        if seq:
            return QueryObject(
                query_id=q.query_id,
                query_sequence=seq,
                query_length=len(seq),
                source_db="ncbi_protein",
                source_id=token,
                organism_hint=q.organism_hint,
                taxon_hint=q.taxon_hint,
            )

    raise RuntimeError(f"Could not resolve sequence for query={q.query_id}, input={token}")


print("ID/sequence resolver and identifier enrichment ready.")

ID/sequence resolver and identifier enrichment ready.


In [3]:
# -----------------------------------------------------------------------------
# Search engines: local BLAST and remote NCBI qblast
# -----------------------------------------------------------------------------

def local_blast_available(program: str = "blastp") -> bool:
    return shutil.which(program) is not None


def _select_engine(config: BlastConfig) -> str:
    if config.mode == "local":
        return "local"
    if config.mode == "remote":
        return "remote"

    # auto mode
    if local_blast_available(config.local_program) and config.local_db_path and Path(config.local_db_path).exists():
        return "local"
    return "remote"


def _compute_subject_coverage(aln_len: int, subject_length: Optional[int]) -> float:
    if not subject_length or subject_length <= 0:
        return 0.0
    return (100.0 * aln_len) / float(subject_length)


def _extract_organism_from_title(title: str) -> Optional[str]:
    text = (title or "").strip()
    if not text:
        return None
    # NCBI titles frequently contain organism in final square brackets.
    m = re.search(r"\[([^\[\]]+)\]\s*$", text)
    if m:
        return m.group(1).strip()
    return None


def _normalize_searchio_hits(
    qresult: Any,
    *,
    query_length: int,
    engine: str,
    db_label: str,
    max_hsps: int,
) -> List[HitRecord]:
    out: List[HitRecord] = []
    for hit in qresult:
        hsps = list(hit.hsps)[:max_hsps]
        for hsp in hsps:
            aln_len = int(getattr(hsp, "aln_span", 0) or 0)
            ident_num = int(getattr(hsp, "ident_num", 0) or 0)
            pct_ident = (100.0 * ident_num / aln_len) if aln_len else 0.0

            evalue = float(getattr(hsp, "evalue", 1.0) or 1.0)
            bitscore = float(getattr(hsp, "bitscore", 0.0) or 0.0)
            qcov = (100.0 * aln_len / query_length) if query_length else 0.0

            description = getattr(hit, "description", "")
            subj_len = getattr(hit, "seq_len", None)
            scov = _compute_subject_coverage(aln_len, subj_len)
            organism_name = _extract_organism_from_title(description)

            rec = HitRecord(
                query_id=str(qresult.id),
                subject_id=str(hit.id),
                percent_identity=pct_ident,
                evalue=evalue,
                bit_score=bitscore,
                alignment_length=aln_len,
                query_coverage=qcov,
                subject_coverage=scov,
                taxid=None,
                organism_name=organism_name,
                subject_title=description,
                query_length=query_length,
                subject_length=int(subj_len) if subj_len else None,
                engine=engine,
                db_label=db_label,
            )
            out.append(rec)
    return out


def run_local_blast(query: QueryObject, config: BlastConfig) -> List[HitRecord]:
    if not config.local_db_path:
        raise ValueError("local_db_path must be configured for local BLAST")

    outfmt_fields = "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs staxids sscinames salltitles slen"

    with tempfile.TemporaryDirectory(prefix="blast_local_") as td:
        qpath = Path(td) / "query.fasta"
        opath = Path(td) / "hits.tsv"

        rec = SeqRecord(Seq(query.query_sequence), id=query.query_id, description="")
        SeqIO.write([rec], str(qpath), "fasta")

        cmd = [
            config.local_program,
            "-query", str(qpath),
            "-db", str(config.local_db_path),
            "-evalue", str(config.evalue),
            "-max_target_seqs", str(config.max_hits),
            "-num_threads", str(config.num_threads),
            "-outfmt", outfmt_fields,
            "-out", str(opath),
        ]

        subprocess.run(cmd, check=True)

        if not opath.exists() or opath.stat().st_size == 0:
            return []

        cols = [
            "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen", "qstart", "qend",
            "sstart", "send", "evalue", "bitscore", "qcovs", "staxids", "sscinames", "salltitles", "slen",
        ]
        df = pd.read_csv(opath, sep="\t", header=None, names=cols)

    hits: List[HitRecord] = []
    for _, row in df.iterrows():
        subject_length = int(row["slen"]) if pd.notna(row["slen"]) else None
        aln_len = int(row["length"]) if pd.notna(row["length"]) else 0
        hit = HitRecord(
            query_id=str(row["qseqid"]),
            subject_id=str(row["sseqid"]),
            percent_identity=float(row["pident"]),
            evalue=float(row["evalue"]),
            bit_score=float(row["bitscore"]),
            alignment_length=aln_len,
            query_coverage=float(row["qcovs"]) if pd.notna(row["qcovs"]) else 0.0,
            subject_coverage=_compute_subject_coverage(aln_len, subject_length),
            taxid=str(row["staxids"]) if pd.notna(row["staxids"]) else None,
            organism_name=str(row["sscinames"]) if pd.notna(row["sscinames"]) else None,
            subject_title=str(row["salltitles"]) if pd.notna(row["salltitles"]) else None,
            query_length=query.query_length,
            subject_length=subject_length,
            engine="local",
            db_label=config.db_version_label,
        )
        hits.append(hit)

    return hits


def run_remote_ncbi_qblast(query: QueryObject, config: BlastConfig) -> List[HitRecord]:
    if config.max_hits > 1000:
        print("[WARN] Remote qblast with >1000 hits can be unstable/time-consuming. Local mode is recommended.")

    result_handle = NCBIWWW.qblast(
        program=config.program,
        database=config.database,
        sequence=query.query_sequence,
        hitlist_size=config.max_hits,
        expect=config.evalue,
        format_type="XML",
    )

    xml_text = result_handle.read()
    result_handle.close()
    parsed = SearchIO.parse(io.StringIO(xml_text), "blast-xml")

    all_hits: List[HitRecord] = []
    for qres in parsed:
        all_hits.extend(
            _normalize_searchio_hits(
                qres,
                query_length=query.query_length,
                engine="remote_ncbi_qblast",
                db_label=config.database,
                max_hsps=config.max_hsps,
            )
        )

    time.sleep(config.ncbi_sleep_sec)
    return all_hits


def run_search_engine(query: QueryObject, config: BlastConfig) -> Tuple[str, List[HitRecord]]:
    engine = _select_engine(config)
    if engine == "local":
        return engine, run_local_blast(query, config)
    return engine, run_remote_ncbi_qblast(query, config)


print("Engine layer ready: local + remote + auto-selection.")

Engine layer ready: local + remote + auto-selection.


In [4]:
# -----------------------------------------------------------------------------
# Post-search filtering and Pfam enrichment
# -----------------------------------------------------------------------------

def _compile_patterns(items: Sequence[str], use_regex: bool) -> List[re.Pattern]:
    pats: List[re.Pattern] = []
    for item in items:
        s = (item or "").strip()
        if not s:
            continue
        if use_regex:
            pats.append(re.compile(s, flags=re.IGNORECASE))
        else:
            pats.append(re.compile(re.escape(s), flags=re.IGNORECASE))
    return pats


def _matches_any(text: str, patterns: Sequence[re.Pattern]) -> bool:
    if not patterns:
        return False
    t = text or ""
    return any(p.search(t) for p in patterns)


def _extract_uniprot_candidate(subject_id: str) -> Optional[str]:
    sid = (subject_id or "").strip()
    if not sid:
        return None

    # Common BLAST IDs: sp|P12345|..., tr|Q9ABC1|..., or plain accession
    if "|" in sid:
        parts = sid.split("|")
        if len(parts) >= 2 and _is_uniprot_accession(parts[1]):
            return parts[1]
    if _is_uniprot_accession(sid):
        return sid
    return None


def _extract_pfams_from_uniprot(uniprot_json: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    pfam_ids: List[str] = []
    pfam_names: List[str] = []

    if not uniprot_json:
        return pfam_ids, pfam_names

    xrefs = uniprot_json.get("uniProtKBCrossReferences") or []
    for x in xrefs:
        if str(x.get("database", "")).lower() != "pfam":
            continue
        pid = str(x.get("id", "")).strip()
        if pid:
            pfam_ids.append(pid)
        props = x.get("properties") or []
        for prop in props:
            if str(prop.get("key", "")).lower() == "entry name":
                name = str(prop.get("value", "")).strip()
                if name:
                    pfam_names.append(name)

    return sorted(set(pfam_ids)), sorted(set(pfam_names))


def enrich_hits_with_pfam(hits: List[HitRecord], config: BlastConfig) -> None:
    if not config.enable_pfam_enrichment:
        return

    for hit in hits:
        uni_acc = _extract_uniprot_candidate(hit.subject_id)
        if not uni_acc:
            continue

        if uni_acc in _pfam_cache:
            cached = _pfam_cache[uni_acc]
            hit.pfam_ids = cached.get("pfam_ids", [])
            hit.pfam_names = cached.get("pfam_names", [])
            continue

        uj = _fetch_uniprot_json(uni_acc, config)
        pfam_ids, pfam_names = _extract_pfams_from_uniprot(uj or {})
        hit.pfam_ids = pfam_ids
        hit.pfam_names = pfam_names
        _pfam_cache[uni_acc] = {"pfam_ids": pfam_ids, "pfam_names": pfam_names}


def apply_filters(hits: List[HitRecord], query: QueryObject, config: BlastConfig) -> Tuple[List[HitRecord], List[HitRecord]]:
    keep: List[HitRecord] = []
    reject: List[HitRecord] = []

    include_patterns = _compile_patterns(config.organism_include, config.use_regex_for_organism)
    exclude_patterns = _compile_patterns(config.organism_exclude, config.use_regex_for_organism)

    min_subj_ratio = getattr(config, "min_subject_to_query_length_ratio", None)
    require_subj_len_for_ratio = bool(getattr(config, "require_subject_length_for_ratio", True))

    for hit in hits:
        reasons: List[str] = []

        if hit.evalue > config.evalue:
            reasons.append(f"evalue>{config.evalue}")
        if hit.percent_identity < config.min_identity:
            reasons.append(f"identity<{config.min_identity}")
        if hit.query_coverage < config.min_query_coverage:
            reasons.append(f"qcov<{config.min_query_coverage}")
        if hit.subject_coverage < config.min_subject_coverage:
            reasons.append(f"scov<{config.min_subject_coverage}")
        if hit.alignment_length < config.min_alignment_length:
            reasons.append(f"aln_len<{config.min_alignment_length}")
        if hit.bit_score < config.min_bit_score:
            reasons.append(f"bitscore<{config.min_bit_score}")

        if config.query_len_min is not None and query.query_length < config.query_len_min:
            reasons.append(f"query_len<{config.query_len_min}")
        if config.query_len_max is not None and query.query_length > config.query_len_max:
            reasons.append(f"query_len>{config.query_len_max}")

        if hit.subject_length is not None:
            if config.subject_len_min is not None and hit.subject_length < config.subject_len_min:
                reasons.append(f"subject_len<{config.subject_len_min}")
            if config.subject_len_max is not None and hit.subject_length > config.subject_len_max:
                reasons.append(f"subject_len>{config.subject_len_max}")

        if min_subj_ratio is not None:
            if hit.subject_length is None:
                if require_subj_len_for_ratio:
                    reasons.append("subject_len_missing_for_ratio")
            else:
                ratio = float(hit.subject_length) / float(query.query_length) if query.query_length else 0.0
                if ratio < float(min_subj_ratio):
                    reasons.append(f"subject_to_query_ratio<{min_subj_ratio}")

        if config.tax_include:
            if not hit.taxid or all(t not in str(hit.taxid) for t in config.tax_include):
                reasons.append("tax_not_in_include")
        if config.tax_exclude:
            if hit.taxid and any(t in str(hit.taxid) for t in config.tax_exclude):
                reasons.append("tax_in_exclude")

        org_blob = " ".join([hit.organism_name or "", hit.subject_title or ""]).strip()
        if include_patterns and not _matches_any(org_blob, include_patterns):
            reasons.append("organism_not_in_include")
        if exclude_patterns and _matches_any(org_blob, exclude_patterns):
            reasons.append("organism_in_exclude")

        if config.pfam_include:
            all_pfam = set(hit.pfam_ids + hit.pfam_names)
            if not any((needle in all_pfam) for needle in config.pfam_include):
                reasons.append("pfam_not_in_include")
        if config.pfam_exclude:
            all_pfam = set(hit.pfam_ids + hit.pfam_names)
            if any((needle in all_pfam) for needle in config.pfam_exclude):
                reasons.append("pfam_in_exclude")

        if reasons:
            hit.passed_filters = False
            hit.filter_reason = ";".join(reasons)
            reject.append(hit)
        else:
            hit.passed_filters = True
            hit.filter_reason = "passed"
            keep.append(hit)

    # rank and cap top-k if needed
    keep_sorted = sorted(keep, key=lambda h: (h.evalue, -h.bit_score, -h.percent_identity))
    if len(keep_sorted) > config.max_hits:
        overflow = keep_sorted[config.max_hits:]
        for h in overflow:
            h.passed_filters = False
            h.filter_reason = "trimmed_by_max_hits"
        reject.extend(overflow)
        keep_sorted = keep_sorted[: config.max_hits]

    return keep_sorted, reject


print("Filter layer ready: blast, taxonomy/organism, length, Pfam.")

Filter layer ready: blast, taxonomy/organism, length, Pfam.


In [5]:
# -----------------------------------------------------------------------------
# Export and orchestration
# -----------------------------------------------------------------------------

def _get_dedup_key_value(hit: HitRecord, key_type: str) -> Optional[str]:
    """Extract the value to use for cross-query deduplication."""
    if key_type == "protein_id":
        return clean_hit_subject_id(hit.subject_id)
    if key_type == "subject_id":
        return str(hit.subject_id or "").strip() or None
    if key_type == "clean_subject_id":
        return clean_hit_subject_id(hit.subject_id)
    return None


def _deduplicate_hits_across_queries(
    hits: Sequence[HitRecord],
    config: BlastConfig,
) -> Tuple[List[HitRecord], Dict[str, Any]]:
    """
    Annotate cross-query duplicates and optionally remove them.

    Returns:
        hits_to_output: tracked-only or deduplicated hits depending on config
        dedup_meta: stats about keys, groups, and removed rows
    """
    key_to_hits: Dict[str, List[HitRecord]] = {}
    unkeyed_hits: List[HitRecord] = []

    for hit in hits:
        key_val = _get_dedup_key_value(hit, config.duplicate_key)
        if not key_val:
            hit.cross_query_duplicate_ids = []
            unkeyed_hits.append(hit)
            continue
        key_to_hits.setdefault(key_val, []).append(hit)

    tracked_hits: List[HitRecord] = []
    deduplicated_hits: List[HitRecord] = []
    duplicate_groups = 0
    duplicate_rows_removed = 0

    for key_val, group in key_to_hits.items():
        query_ids = sorted({str(h.source_query_id).strip() for h in group if str(h.source_query_id or "").strip()})

        # Add duplicate annotations to each hit in the group.
        for h in group:
            source_q = str(h.source_query_id or "").strip()
            h.cross_query_duplicate_ids = [q for q in query_ids if q != source_q]
            tracked_hits.append(h)

        # Always keep one representative in dedup mode.
        best_hit = min(group, key=lambda h: (h.evalue, -h.bit_score, -h.percent_identity))
        deduplicated_hits.append(best_hit)

        if len(group) > 1:
            duplicate_groups += 1
            duplicate_rows_removed += len(group) - 1

    tracked_hits.extend(unkeyed_hits)
    deduplicated_hits.extend(unkeyed_hits)

    tracked_hits.sort(key=lambda h: (h.evalue, -h.bit_score, -h.percent_identity))
    deduplicated_hits.sort(key=lambda h: (h.evalue, -h.bit_score, -h.percent_identity))

    output_hits = deduplicated_hits if config.deduplicate_across_queries else tracked_hits
    dedup_meta = {
        "unique_keys": len(key_to_hits),
        "unkeyed_hits": len(unkeyed_hits),
        "duplicate_groups": duplicate_groups,
        "duplicate_rows_removed": duplicate_rows_removed,
    }
    return output_hits, dedup_meta


def build_resolver_first_df(hits: Sequence[HitRecord], config: BlastConfig) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for idx, hit in enumerate(hits, start=1):
        enriched = enrich_hit_identifiers(hit, config)

        clean_subject_id = enriched.get("clean_subject_id") or clean_hit_subject_id(hit.subject_id)
        protein_id = enriched.get("protein_id") or clean_subject_id
        uniprot_accession = enriched.get("uniprot_accession")
        locus_tag_candidate = enriched.get("locus_tag_candidate")
        genome_accession_candidate = enriched.get("genome_accession_candidate")

        out_hit_id = genome_accession_candidate or clean_subject_id
        out_accession_id = locus_tag_candidate or uniprot_accession or protein_id

        source_query_id = str(hit.source_query_id or "").strip()
        other_queries = sorted({str(q).strip() for q in (hit.cross_query_duplicate_ids or []) if str(q).strip()})

        all_queries: List[str] = []
        if source_query_id:
            all_queries.append(source_query_id)
        for q in other_queries:
            if q not in all_queries:
                all_queries.append(q)

        duplicate_count = len(other_queries)
        duplicate_group_size = len(all_queries) if all_queries else 1

        rows.append(
            {
                "ssn_id": 0,
                "cluster_id": idx,
                "radial_rank_outward": 1,
                "radial_distance": float(hit.evalue),
                "hit_id": out_hit_id,
                "organism": hit.organism_name or "",
                "accession_id": out_accession_id,
                "subject_id_raw": hit.subject_id,
                "subject_id_clean": clean_subject_id,
                "protein_id": protein_id,
                "uniprot_accession": uniprot_accession,
                "locus_tag_candidate": locus_tag_candidate,
                "genome_accession_candidate": genome_accession_candidate,
                "gene_names": enriched.get("gene_names"),
                "orf_names": enriched.get("orf_names"),
                "ordered_locus_names": enriched.get("ordered_locus_names"),
                "refseq_xrefs": enriched.get("refseq_ids"),
                "embl_xrefs": enriched.get("embl_ids"),
                "percent_identity": hit.percent_identity,
                "evalue": hit.evalue,
                "bit_score": hit.bit_score,
                "query_coverage": hit.query_coverage,
                "subject_coverage": hit.subject_coverage,
                "alignment_length": hit.alignment_length,
                "taxid": hit.taxid,
                "subject_title": hit.subject_title,
                "query_length": hit.query_length,
                "subject_length": hit.subject_length,
                "engine": hit.engine,
                "db_label": hit.db_label,
                "pfam_ids": ";".join(hit.pfam_ids),
                "pfam_names": ";".join(hit.pfam_names),
                "filter_status": hit.filter_reason,
                "source_query_id": source_query_id,
                "is_cross_query_duplicate": duplicate_count > 0,
                "other_queries_with_hit": ";".join(other_queries),
                "other_queries_count": duplicate_count,
                "duplicate_group_size": duplicate_group_size,
                "all_queries_with_hit": ";".join(all_queries),
            }
        )
    return pd.DataFrame(rows)


def write_run_outputs(
    *,
    kept_hits: Sequence[HitRecord],
    rejected_hits: Sequence[HitRecord],
    config: BlastConfig,
    run_meta: Dict[str, Any],
) -> Dict[str, Path]:
    stamp = now_str()
    prefix = f"{config.output_prefix}_{stamp}"

    resolver_csv = OUTPUT_DIR / f"{prefix}_resolver_input.csv"
    audit_csv = OUTPUT_DIR / f"{prefix}_filter_audit.csv"
    meta_json = OUTPUT_DIR / f"{prefix}_run_meta.json"

    kept_df = build_resolver_first_df(kept_hits, config)
    kept_df.to_csv(resolver_csv, sep=";", index=False)

    if config.write_filter_audit_csv:
        audit_df = pd.DataFrame([as_hit_dict(h) for h in [*kept_hits, *rejected_hits]])
        audit_df.to_csv(audit_csv, sep=";", index=False)

    with open(meta_json, "w", encoding="utf-8") as fh:
        json.dump(run_meta, fh, indent=2)

    return {
        "resolver_csv": resolver_csv,
        "audit_csv": audit_csv,
        "meta_json": meta_json,
    }


def run_hybrid_blast_search(
    query_inputs: Sequence[QueryInput],
    config: BlastConfig,
) -> Dict[str, Any]:
    config.validate()

    if not config.entrez_email:
        raise ValueError("entrez_email is required")

    resolved_queries: List[QueryObject] = [resolve_query_input(q, config) for q in query_inputs]

    all_raw_hits: List[HitRecord] = []
    all_kept_hits: List[HitRecord] = []
    all_rejected_hits: List[HitRecord] = []

    per_query_stats: List[Dict[str, Any]] = []
    selected_engine = None

    # Only enrich pre-filter when Pfam filters are active.
    needs_pfam_for_filter = bool(config.pfam_include or config.pfam_exclude)

    for query in resolved_queries:
        engine, raw_hits = run_search_engine(query, config)
        selected_engine = selected_engine or engine
        all_raw_hits.extend(raw_hits)

        if needs_pfam_for_filter:
            enrich_hits_with_pfam(raw_hits, config)

        kept, rejected = apply_filters(raw_hits, query, config)

        for hit in kept:
            hit.source_query_id = query.query_id
        for hit in rejected:
            hit.source_query_id = query.query_id

        all_kept_hits.extend(kept)
        all_rejected_hits.extend(rejected)

        per_query_stats.append(
            {
                "query_id": query.query_id,
                "query_length": query.query_length,
                "source_db": query.source_db,
                "source_id": query.source_id,
                "raw_hits": len(raw_hits),
                "kept_hits": len(kept),
                "rejected_hits": len(rejected),
            }
        )

    final_kept_hits = all_kept_hits
    dedup_stats: Dict[str, Any] = {}

    if len(resolved_queries) > 1:
        pre_dedup_count = len(all_kept_hits)
        final_kept_hits, dedup_meta = _deduplicate_hits_across_queries(all_kept_hits, config)

        dedup_stats = {
            "queries_count": len(resolved_queries),
            "pre_dedup_kept_hits": pre_dedup_count,
            "post_dedup_kept_hits": len(final_kept_hits),
            "duplicates_detected": int(dedup_meta["duplicate_rows_removed"]),
            "duplicates_removed": int(dedup_meta["duplicate_rows_removed"]) if config.deduplicate_across_queries else 0,
            "deduplication_enabled": config.deduplicate_across_queries,
            "duplicate_key": config.duplicate_key,
            "duplicate_groups": int(dedup_meta["duplicate_groups"]),
            "unkeyed_hits": int(dedup_meta["unkeyed_hits"]),
        }
    else:
        for hit in final_kept_hits:
            hit.cross_query_duplicate_ids = []

    # If Pfam is needed for output but not for filtering, enrich only final hits.
    if config.enable_pfam_enrichment and not needs_pfam_for_filter:
        enrich_hits_with_pfam(final_kept_hits, config)

    run_meta = {
        "timestamp": datetime.now().isoformat(),
        "engine": selected_engine,
        "mode": config.mode,
        "remote_provider": config.remote_provider,
        "db_version_label": config.db_version_label,
        "config": asdict(config),
        "per_query_stats": per_query_stats,
        "dedup_stats": dedup_stats,
        "total_raw_hits": len(all_raw_hits),
        "total_kept_hits_pre_dedup": len(all_kept_hits),
        "total_kept_hits_post_dedup": len(final_kept_hits),
        "total_rejected_hits": len(all_rejected_hits),
    }

    output_paths = write_run_outputs(
        kept_hits=final_kept_hits,
        rejected_hits=all_rejected_hits,
        config=config,
        run_meta=run_meta,
    )

    print("Run summary:")
    print(f"  Engine: {selected_engine}")
    print(f"  Raw hits: {len(all_raw_hits)}")
    print(f"  Kept hits (pre-dedup): {len(all_kept_hits)}")
    print(f"  Kept hits (output): {len(final_kept_hits)}")
    print(f"  Rejected hits: {len(all_rejected_hits)}")
    if len(resolved_queries) > 1:
        if dedup_stats.get("duplicates_detected", 0) > 0:
            print(f"  Cross-query duplicates detected: {dedup_stats['duplicates_detected']}")
            print(f"  Duplicate groups: {dedup_stats.get('duplicate_groups', 0)}")
        if config.deduplicate_across_queries:
            print(f"  Duplicates removed: {dedup_stats.get('duplicates_removed', 0)}")
    print(f"  Resolver CSV: {output_paths['resolver_csv']}")
    print(f"  Audit CSV: {output_paths['audit_csv']}")
    print(f"  Meta JSON: {output_paths['meta_json']}")

    return {
        "queries": resolved_queries,
        "raw_hits": all_raw_hits,
        "kept_hits": final_kept_hits,
        "rejected_hits": all_rejected_hits,
        "run_meta": run_meta,
        "output_paths": output_paths,
    }


print("Orchestration ready: run_hybrid_blast_search(...)")

Orchestration ready: run_hybrid_blast_search(...)


In [6]:
# -----------------------------------------------------------------------------
# Usage (configured test run)
# -----------------------------------------------------------------------------

config = BlastConfig(
    mode="auto",
    remote_provider="ncbi_qblast",
    local_program="blastp",
    local_db_path=None,  # set to local DB path for local mode
    db_version_label="nr_live_or_local_snapshot",
    evalue=1e-5,
    max_hits=200,
    max_hsps=1,
    min_identity=30.0,
    min_query_coverage=50.0,
    min_alignment_length=80,
    min_bit_score=50.0,
    tax_include=[],
    tax_exclude=[],
    organism_include=[],
    # Exclude non-concrete organism names like "Genus sp." and "... bacterium/bacteria"
    organism_exclude=[r"\bsp\.", r"\bbacteri(?:um|a)\b"],
    use_regex_for_organism=True,
    enable_pfam_enrichment=True,
    pfam_include=[],
    pfam_exclude=[],
    entrez_email="phitro@bu.edu",
    output_prefix="enzyme_homologue_blast_test",
)

# Require subject sequence length >= 80% of the query amino acid length.
config.min_subject_to_query_length_ratio = 0.8
config.require_subject_length_for_ratio = True

queries = [
    QueryInput(query_id="wp_214186582_1", input_id="WP_214186582.1"),
    QueryInput(query_id="a0a9w6g2y3", input_id="A0A9W6G2Y3"),
]

print("Prepared configured test run with 2 IDs and max_hits=200")
result = run_hybrid_blast_search(queries, config)
print("Test run finished.")

Prepared configured test run with 2 IDs and max_hits=200
Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits (pre-dedup): 0
  Kept hits (output): 0
  Rejected hits: 400
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_20260327_224924_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_20260327_224924_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_20260327_224924_run_meta.json
Test run finished.


In [7]:
# Quick audit diagnosis for the latest run
from pathlib import Path

latest_audit = sorted(OUTPUT_DIR.glob("enzyme_homologue_blast_test_*_filter_audit.csv"))[-1]
audit_df = pd.read_csv(latest_audit, sep=";")
print("Latest audit:", latest_audit)
print("Rows:", len(audit_df))
print(audit_df["filter_reason"].value_counts().head(15))

Latest audit: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_20260327_224924_filter_audit.csv
Rows: 400
filter_reason
evalue>1e-05;organism_in_exclude    302
evalue>1e-05                         98
Name: count, dtype: int64


In [8]:
# Optional diagnostics: length availability in latest run
print("subject_length missing:", int(audit_df["subject_length"].isna().sum()))
print("subject_length present:", int(audit_df["subject_length"].notna().sum()))

subject_length missing: 0
subject_length present: 400


In [9]:
# Rerun with relaxed e-value while keeping the requested filters
config.evalue = 1.0
config.output_prefix = "enzyme_homologue_blast_test_e1"

print("Rerun with evalue=1.0, max_hits=200, 80% length ratio, and organism exclusions")
result_e1 = run_hybrid_blast_search(queries, config)
print("Rerun finished.")

Rerun with evalue=1.0, max_hits=200, 80% length ratio, and organism exclusions
Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits (pre-dedup): 98
  Kept hits (output): 98
  Rejected hits: 302
  Cross-query duplicates detected: 49
  Duplicate groups: 49
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_e1_20260327_230504_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_e1_20260327_230504_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_test_e1_20260327_230504_run_meta.json
Rerun finished.


In [10]:
# Validate the kept set against requested organism exclusion patterns
kept_df = pd.read_csv(result_e1["output_paths"]["resolver_csv"], sep=";")
org_series = kept_df["organism"].fillna("").astype(str)
violations = kept_df[
    org_series.str.contains(r"\bsp\.", case=False, regex=True)
    | org_series.str.contains(r"\bbacteri(?:um|a)\b", case=False, regex=True)
]
print("Kept rows:", len(kept_df))
print("Rows violating exclusion pattern:", len(violations))
print(kept_df[["accession_id", "organism"]].head(10))

Kept rows: 98
Rows violating exclusion pattern: 0
     accession_id                   organism
0  GHYDROH2_29460  Geobacter hydrogenophilus
1  GHYDROH2_29460  Geobacter hydrogenophilus
2     JZM60_01270  Geobacter benzoatilyticus
3     JZM60_01270  Geobacter benzoatilyticus
4     GPICK_07515      Geobacter pickeringii
5     GPICK_07515      Geobacter pickeringii
6     ET418_06540           Oryzomonas rubra
7     ET418_06540           Oryzomonas rubra
8      JN12_02524      Geobacter argillaceus
9      JN12_02524      Geobacter argillaceus


In [11]:
# Refined rerun with updated hit/accession mapping
config.evalue = 1.0
config.output_prefix = "enzyme_homologue_blast_refined"

print("Running refined mapping test...")
result_refined = run_hybrid_blast_search(queries, config)
print("Refined run finished.")

Running refined mapping test...
Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits (pre-dedup): 98
  Kept hits (output): 98
  Rejected hits: 302
  Cross-query duplicates detected: 49
  Duplicate groups: 49
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_20260327_231947_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_20260327_231947_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_20260327_231947_run_meta.json
Refined run finished.


In [12]:
# Validate refined identifier mapping output
refined_df = pd.read_csv(result_refined["output_paths"]["resolver_csv"], sep=";")

print("Rows:", len(refined_df))
print("Columns:", ", ".join(refined_df.columns[:18]), "...")

# 1) check if raw ref|...| is gone from exported hit_id in most rows
has_pipe_in_hit = refined_df["hit_id"].astype(str).str.contains(r"\|").sum()
print("hit_id entries containing '|':", int(has_pipe_in_hit))

# 2) show mapping preview
preview_cols = [
    "hit_id",
    "accession_id",
    "subject_id_raw",
    "subject_id_clean",
    "protein_id",
    "uniprot_accession",
    "locus_tag_candidate",
    "genome_accession_candidate",
    "organism",
]
print(refined_df[preview_cols].head(15).to_string(index=False))

# 3) summary counts for preferred mappings
genome_in_hit = refined_df["genome_accession_candidate"].notna().sum()
locus_in_accession = refined_df["locus_tag_candidate"].notna().sum()
uniprot_in_accession = refined_df["uniprot_accession"].notna().sum()
print("Rows with genome accession candidate:", int(genome_in_hit))
print("Rows with locus tag candidate:", int(locus_in_accession))
print("Rows with UniProt accession candidate:", int(uniprot_in_accession))

Rows: 98
Columns: ssn_id, cluster_id, radial_rank_outward, radial_distance, hit_id, organism, accession_id, subject_id_raw, subject_id_clean, protein_id, uniprot_accession, locus_tag_candidate, genome_accession_candidate, gene_names, orf_names, ordered_locus_names, refseq_xrefs, embl_xrefs ...
hit_id entries containing '|': 0
         hit_id   accession_id      subject_id_raw subject_id_clean     protein_id uniprot_accession locus_tag_candidate genome_accession_candidate                  organism
 WP_214186582.1 GHYDROH2_29460 ref|WP_214186582.1|   WP_214186582.1 WP_214186582.1        A0A9W6G2Y3      GHYDROH2_29460                        NaN Geobacter hydrogenophilus
 WP_214186582.1 GHYDROH2_29460 ref|WP_214186582.1|   WP_214186582.1 WP_214186582.1        A0A9W6G2Y3      GHYDROH2_29460                        NaN Geobacter hydrogenophilus
    NZ_CP071382    JZM60_01270 ref|WP_207163745.1|   WP_207163745.1 WP_207163745.1        A0ABX7Q3C5         JZM60_01270                NZ_CP071382 Ge

In [13]:
# Validate generic-organism exclusion still holds in refined run
org_series_refined = refined_df["organism"].fillna("").astype(str)
viol_refined = refined_df[
    org_series_refined.str.contains(r"\bsp\.", case=False, regex=True)
    | org_series_refined.str.contains(r"\bbacteri(?:um|a)\b", case=False, regex=True)
]
print("Refined rows violating organism exclusion:", len(viol_refined))

Refined rows violating organism exclusion: 0


In [14]:
# Rerun after adding special-case resolver step
config.evalue = 1.0
config.output_prefix = "enzyme_homologue_blast_refined_special"

print("Running special-case resolver rerun...")
result_refined_special = run_hybrid_blast_search(queries, config)
print("Special-case rerun finished.")

Running special-case resolver rerun...


c:\Users\poker\AppData\Local\Programs\Python\Python311\Lib\site-packages\Bio\Blast\NCBIWWW.py:275: BiopythonWarning: BLAST request WEJ79TAA016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits (pre-dedup): 98
  Kept hits (output): 98
  Rejected hits: 302
  Cross-query duplicates detected: 49
  Duplicate groups: 49
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_special_20260327_233433_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_special_20260327_233433_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_special_20260327_233433_run_meta.json
Special-case rerun finished.


In [15]:
# Compare refined vs special-case resolver coverage
special_df = pd.read_csv(result_refined_special["output_paths"]["resolver_csv"], sep=";")

special_genome = int(special_df["genome_accession_candidate"].notna().sum())
special_locus = int(special_df["locus_tag_candidate"].notna().sum())
special_uniprot = int(special_df["uniprot_accession"].notna().sum())

print("Special-case run rows:", len(special_df))
print("Rows with genome accession candidate:", special_genome)
print("Rows with locus tag candidate:", special_locus)
print("Rows with UniProt accession candidate:", special_uniprot)

remaining_missing = special_df[
    special_df["genome_accession_candidate"].isna() | special_df["locus_tag_candidate"].isna()
][["subject_id_clean", "protein_id", "genome_accession_candidate", "locus_tag_candidate", "organism"]]
print("Rows still missing genome or locus candidate:", len(remaining_missing))
print(remaining_missing.head(12).to_string(index=False))

Special-case run rows: 98
Rows with genome accession candidate: 90
Rows with locus tag candidate: 74
Rows with UniProt accession candidate: 58
Rows still missing genome or locus candidate: 32
subject_id_clean     protein_id genome_accession_candidate locus_tag_candidate                   organism
  WP_214186582.1 WP_214186582.1                        NaN      GHYDROH2_29460  Geobacter hydrogenophilus
  WP_214186582.1 WP_214186582.1                        NaN      GHYDROH2_29460  Geobacter hydrogenophilus
  WP_281184841.1 WP_281184841.1                NZ_CM144228                 NaN    Trichlorobacter lovleyi
  WP_281184841.1 WP_281184841.1                NZ_CM144228                 NaN    Trichlorobacter lovleyi
  WP_085814426.1 WP_085814426.1                        NaN       GPEL0_01r4534 Geoanaerobacter pelophilus
  WP_085814426.1 WP_085814426.1                        NaN       GPEL0_01r4534 Geoanaerobacter pelophilus
  WP_136524231.1 WP_136524231.1            NZ_SSYA01000001        

In [16]:
# Quality checks for special-case run
special_org = special_df["organism"].fillna("").astype(str)
special_viol = special_df[
    special_org.str.contains(r"\bsp\.", case=False, regex=True)
    | special_org.str.contains(r"\bbacteri(?:um|a)\b", case=False, regex=True)
]

pipes_in_hit_id = int(special_df["hit_id"].astype(str).str.contains(r"\|").sum())
print("Special-case run organism filter violations:", len(special_viol))
print("Special-case run hit_id entries containing '|':", pipes_in_hit_id)
print(special_df[["hit_id", "accession_id", "subject_id_clean", "genome_accession_candidate", "locus_tag_candidate"]].head(10).to_string(index=False))

Special-case run organism filter violations: 0
Special-case run hit_id entries containing '|': 0
         hit_id   accession_id subject_id_clean genome_accession_candidate locus_tag_candidate
 WP_214186582.1 GHYDROH2_29460   WP_214186582.1                        NaN      GHYDROH2_29460
 WP_214186582.1 GHYDROH2_29460   WP_214186582.1                        NaN      GHYDROH2_29460
    NZ_CP071382    JZM60_01270   WP_207163745.1                NZ_CP071382         JZM60_01270
    NZ_CP071382    JZM60_01270   WP_207163745.1                NZ_CP071382         JZM60_01270
    NZ_CP009788    GPICK_07515   WP_039741821.1                NZ_CP009788         GPICK_07515
    NZ_CP009788    GPICK_07515   WP_039741821.1                NZ_CP009788         GPICK_07515
NZ_SRSD01000003    ET418_06540   WP_149306776.1            NZ_SRSD01000003         ET418_06540
NZ_SRSD01000003    ET418_06540   WP_149306776.1            NZ_SRSD01000003         ET418_06540
NZ_VLLN01000015     JN12_02524   WP_145023268.1 

In [26]:
# Rerun after adding genome feature search resolver step
config.evalue = 1.0
config.output_prefix = "enzyme_homologue_blast_refined_featuresearch"

print("Running genome-feature-search resolver rerun...")
result_refined_featuresearch = run_hybrid_blast_search(queries, config)
print("Genome-feature-search rerun finished.")

Running genome-feature-search resolver rerun...


c:\Users\poker\AppData\Local\Programs\Python\Python311\Lib\site-packages\Bio\Blast\NCBIWWW.py:275: BiopythonWarning: BLAST request WDSGM2RH016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits: 98
  Rejected hits: 302
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_featuresearch_20260327_162044_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_featuresearch_20260327_162044_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_refined_featuresearch_20260327_162044_run_meta.json
Genome-feature-search rerun finished.


In [27]:
# Compare special-case vs genome-feature-search coverage
feature_df = pd.read_csv(result_refined_featuresearch["output_paths"]["resolver_csv"], sep=";")

feature_genome = int(feature_df["genome_accession_candidate"].notna().sum())
feature_locus = int(feature_df["locus_tag_candidate"].notna().sum())
feature_uniprot = int(feature_df["uniprot_accession"].notna().sum())

print("Feature-search run rows:", len(feature_df))
print("Rows with genome accession candidate:", feature_genome)
print("Rows with locus tag candidate:", feature_locus)
print("Rows with UniProt accession candidate:", feature_uniprot)

if "special_df" in globals():
    print("Delta vs previous special run (feature - special):")
    print("  genome candidates delta:", feature_genome - int(special_df["genome_accession_candidate"].notna().sum()))
    print("  locus candidates delta:", feature_locus - int(special_df["locus_tag_candidate"].notna().sum()))

# quality checks
feature_org = feature_df["organism"].fillna("").astype(str)
feature_viol = feature_df[
    feature_org.str.contains(r"\bsp\.", case=False, regex=True)
    | feature_org.str.contains(r"\bbacteri(?:um|a)\b", case=False, regex=True)
]
print("Feature-search run organism filter violations:", len(feature_viol))
print("Feature-search run hit_id entries containing '|':", int(feature_df["hit_id"].astype(str).str.contains(r"\|").sum()))

remaining_feature_missing = feature_df[
    feature_df["locus_tag_candidate"].isna()
][["subject_id_clean", "protein_id", "genome_accession_candidate", "locus_tag_candidate", "organism"]]
print("Rows still missing locus tag candidate:", len(remaining_feature_missing))
print(remaining_feature_missing.head(12).to_string(index=False))

Feature-search run rows: 98
Rows with genome accession candidate: 96
Rows with locus tag candidate: 74
Rows with UniProt accession candidate: 58
Delta vs previous special run (feature - special):
  genome candidates delta: 2
  locus candidates delta: 14
Feature-search run organism filter violations: 0
Feature-search run hit_id entries containing '|': 0
Rows still missing locus tag candidate: 24
subject_id_clean     protein_id genome_accession_candidate locus_tag_candidate                        organism
  WP_281184841.1 WP_281184841.1                NZ_CM144228                 NaN         Trichlorobacter lovleyi
  WP_136524231.1 WP_136524231.1            NZ_SSYA01000001                 NaN          Geomonas ferrireducens
  WP_136515210.1 WP_136515210.1            NZ_SSYB01000004                 NaN               Geomonas edaphica
  WP_265551533.1 WP_265551533.1                NZ_CP058409                 NaN         Trichlorobacter lovleyi
  WP_129125118.1 WP_129125118.1            NZ_R

In [9]:
# Test 1: Track cross-query duplicates WITHOUT removing them
config_track_only = BlastConfig(
    mode="auto",
    remote_provider="ncbi_qblast",
    evalue=1.0,
    max_hits=200,
    min_identity=30.0,
    min_query_coverage=50.0,
    min_alignment_length=80,
    min_bit_score=50.0,
    organism_exclude=[r"\bsp\.", r"\bbacteri(?:um|a)\b"],
    use_regex_for_organism=True,
    enable_pfam_enrichment=True,
    # DEDUPLICATION SETTINGS
    deduplicate_across_queries=False,  # Track but don't remove duplicates
    duplicate_key="protein_id",
    entrez_email="phitro@bu.edu",
    output_prefix="enzyme_homologue_blast_track_duplicates",
)

queries_multi = [
    QueryInput(query_id="wp_214186582_1", input_id="WP_214186582.1"),
    QueryInput(query_id="a0a9w6g2y3", input_id="A0A9W6G2Y3"),
]

print("Test 1: Running BLAST with cross-query duplicate TRACKING (dedup OFF)...")
result_track = run_hybrid_blast_search(queries_multi, config_track_only)
print("Test 1 complete.\n")


Test 1: Running BLAST with cross-query duplicate TRACKING (dedup OFF)...
Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits (pre-dedup): 98
  Kept hits (output): 98
  Rejected hits: 302
  Cross-query duplicates detected: 49
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_track_duplicates_20260327_170841_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_track_duplicates_20260327_170841_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_track_duplicates_20260327_170841_run_meta.json
Test 1 complete.



In [10]:
# Test 2: Remove cross-query duplicates
config_dedup = BlastConfig(
    mode="auto",
    remote_provider="ncbi_qblast",
    evalue=1.0,
    max_hits=200,
    min_identity=30.0,
    min_query_coverage=50.0,
    min_alignment_length=80,
    min_bit_score=50.0,
    organism_exclude=[r"\bsp\.", r"\bbacteri(?:um|a)\b"],
    use_regex_for_organism=True,
    enable_pfam_enrichment=True,
    # DEDUPLICATION SETTINGS
    deduplicate_across_queries=True,  # Remove duplicates, keep best by evalue/bitscore
    duplicate_key="protein_id",
    entrez_email="phitro@bu.edu",
    output_prefix="enzyme_homologue_blast_deduplicated",
)

print("Test 2: Running BLAST with cross-query duplicate REMOVAL (dedup ON)...")
result_dedup = run_hybrid_blast_search(queries_multi, config_dedup)
print("Test 2 complete.\n")


Test 2: Running BLAST with cross-query duplicate REMOVAL (dedup ON)...


c:\Users\poker\AppData\Local\Programs\Python\Python311\Lib\site-packages\Bio\Blast\NCBIWWW.py:275: BiopythonWarning: BLAST request WDWEX88F016 is taking longer than 10 minutes, consider re-issuing it
  warnings.warn(


Run summary:
  Engine: remote
  Raw hits: 400
  Kept hits (pre-dedup): 98
  Kept hits (output): 49
  Rejected hits: 302
  Duplicates removed: 49
  Resolver CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_deduplicated_20260327_173420_resolver_input.csv
  Audit CSV: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_deduplicated_20260327_173420_filter_audit.csv
  Meta JSON: D:\Studium\PhD\Rotation\1st_WetLab\Enzyme_Homologe_Search\blast_outputs\enzyme_homologue_blast_deduplicated_20260327_173420_run_meta.json
Test 2 complete.



In [ ]:
# Compare tracking vs deduplication results
track_df = pd.read_csv(result_track["output_paths"]["resolver_csv"], sep=";")
dedup_df = pd.read_csv(result_dedup["output_paths"]["resolver_csv"], sep=";")

print("=" * 80)
print("COMPARISON: Tracking vs Deduplication")
print("=" * 80)
print(f"\nWith tracking (dedup OFF):  {len(track_df)} rows")
print(f"With deduplication (ON):    {len(dedup_df)} rows")
print(f"Duplicates detected:        {len(track_df) - len(dedup_df)} rows")

print("\n--- Cross-Query Duplicate Tracking ---")
cross_query_hits = track_df[track_df["is_cross_query_duplicate"] == True]
print(f"Rows found in multiple queries: {len(cross_query_hits)}")

if len(cross_query_hits) > 0:
    print("\nExamples of cross-query duplicates:")
    example_cols = [
        "source_query_id",
        "protein_id",
        "hit_id",
        "accession_id",
        "is_cross_query_duplicate",
        "other_queries_with_hit",
        "evalue",
    ]
    print(cross_query_hits[example_cols].head(10).to_string(index=False))

print("\n--- Deduplication Impact ---")
print(f"Best entries kept per protein: {len(dedup_df)}")
print(f"Removed redundant copies: {len(track_df) - len(dedup_df)}")

# Show query distribution
print("\n--- Query Distribution (Tracking Mode) ---")
query_counts_track = track_df["source_query_id"].value_counts().to_dict()
for qid, count in sorted(query_counts_track.items()):
    print(f"  {qid}: {count} hits")

print("\n--- Query Distribution (Dedup Mode) ---")
query_counts_dedup = dedup_df["source_query_id"].value_counts().to_dict()
for qid, count in sorted(query_counts_dedup.items()):
    print(f"  {qid}: {count} hits (after dedup)")


COMPARISON: Tracking vs Deduplication

With tracking (dedup OFF):  98 rows
With deduplication (ON):    49 rows
Duplicates detected:        49 rows

--- Cross-Query Duplicate Tracking ---
Rows found in multiple queries: 98

Examples of cross-query duplicates:
source_query_id     protein_id          hit_id   accession_id  is_cross_query_duplicate other_queries_with_hit  evalue
 wp_214186582_1 WP_214186582.1 NZ_BSDS01000002 GHYDROH2_29460                      True             a0a9w6g2y3     1.0
     a0a9w6g2y3 WP_214186582.1 NZ_BSDS01000002 GHYDROH2_29460                      True         wp_214186582_1     1.0
 wp_214186582_1 WP_207163745.1  WP_207163745.1    JZM60_01270                      True             a0a9w6g2y3     1.0
     a0a9w6g2y3 WP_207163745.1  WP_207163745.1    JZM60_01270                      True         wp_214186582_1     1.0
 wp_214186582_1 WP_039741821.1  WP_039741821.1    GPICK_07515                      True             a0a9w6g2y3     1.0
     a0a9w6g2y3 WP_03974182